In [3]:
import pandas as pd
import re
import nltk
from nltk.corpus import stopwords
import gensim
from gensim import corpora
from gensim.models import LdaModel

nltk.download("stopwords", quiet=True)
STOPWORDS = set(stopwords.words("english"))


df = pd.read_csv("../data/processed/sentiment140_sample.csv", encoding="latin-1")
print(df.shape)

(50000, 6)


In [4]:
CUSTOM_STOPWORDS = STOPWORDS.union({
    "amp", "dont", "cant", "wont", "im", "ive", "youre", "thats",
    "get", "got", "like", "would", "really", "one", "know", "want",
    "go", "going", "still", "much", "lol"
})

def clean_for_topics(text):
    if not isinstance(text, str):
        return []
    text = text.lower()
    text = re.sub(r"http\S+|www\S+", "", text)
    text = re.sub(r"@\w+", "", text)
    text = re.sub(r"&amp;?", "", text)
    text = re.sub(r"[^a-z\s]", "", text)
    words = text.split()
    words = [w for w in words if w not in CUSTOM_STOPWORDS and len(w) > 2]
    return words

df["tokens"] = df["text"].apply(clean_for_topics)
df = df[df["tokens"].map(len) > 0].reset_index(drop=True)

print(df.shape)

(49616, 7)


In [5]:
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.decomposition import LatentDirichletAllocation
import joblib

# Reuse df["tokens"] from earlier — join back into strings for CountVectorizer
df["joined_tokens"] = df["tokens"].apply(lambda tokens: " ".join(tokens))

vectorizer = CountVectorizer(max_features=5000)
doc_term_matrix = vectorizer.fit_transform(df["joined_tokens"])

NUM_TOPICS = 6
lda_model = LatentDirichletAllocation(
    n_components=NUM_TOPICS,
    random_state=42,
    max_iter=20
)
lda_model.fit(doc_term_matrix)

# Print top words per topic to confirm it matches similar quality to before
feature_names = vectorizer.get_feature_names_out()
for idx, topic in enumerate(lda_model.components_):
    top_words = [feature_names[i] for i in topic.argsort()[-8:][::-1]]
    print(f"Topic {idx}: {', '.join(top_words)}")

Topic 0: work, miss, back, happy, way, wait, home, tonight
Topic 1: haha, hey, love, tired, omg, havent, yet, saw
Topic 2: good, thanks, sorry, twitter, yeah, thank, well, people
Topic 3: day, today, home, morning, work, school, sad, sick
Topic 4: night, see, wish, last, well, love, watching, good
Topic 5: new, need, please, love, sure, find, yes, looking


In [6]:
joblib.dump(lda_model, "../trained_models/lda_model_sklearn.pkl")
joblib.dump(vectorizer, "../trained_models/lda_vectorizer_sklearn.pkl")

topic_labels = {
    0: "Daily Routine / Time",
    1: "Apology / Hesitation",
    2: "Entertainment / Movies",
    3: "Affection / Wishing",
    4: "Work / Daily Life",
    5: "Mixed Mood / Reactions"
}
joblib.dump(topic_labels, "../trained_models/lda_topic_labels.pkl")

print("Saved sklearn LDA model, vectorizer, and labels.")

Saved sklearn LDA model, vectorizer, and labels.
